# InsightFlow Executive — Fine-tuning NLP Models
**Fine-tune RoBERTa + XLM-RoBERTa on InsightFlow synthetic dataset**

- Dataset : 6000 messages (3000 EN + 3000 FR)
- Models  : RoBERTa Twitter + XLM-RoBERTa Multilingual
- GPU     : T4 (free Colab)
- Time    : ~20-30 min per model

In [ ]:
# ── Step 1 : Install dependencies ────────────────────────────
!pip install -q transformers datasets torch scikit-learn accelerate evaluate

In [ ]:
# ── Step 2 : Upload dataset files from your machine ──────────
from google.colab import files
print('Upload your dataset files:')
print('  - insightflow_synthetic_en.csv')
print('  - insightflow_synthetic_fr.csv  (when ready)')
print('  - insightflow_synthetic_full.csv (when both ready)')
uploaded = files.upload()

In [ ]:
# ── Step 3 : Upload finetune.py ──────────────────────────────
print('Now upload ml/finetune.py')
uploaded2 = files.upload()

In [ ]:
# ── Step 4 : Setup folder structure ──────────────────────────
import os
os.makedirs('ml/dataset', exist_ok=True)
os.makedirs('ml/models', exist_ok=True)

# Move uploaded files
for fname in uploaded.keys():
    os.rename(fname, f'ml/dataset/{fname}')
    print(f'Moved: {fname} → ml/dataset/')

os.rename('finetune.py', 'ml/finetune.py')
print('Setup complete.')

In [ ]:
# ── Step 5a : Fine-tune XLM-RoBERTa (FR+EN) ← RECOMMENDED ───
!python ml/finetune.py --model xlm --lang en
# Change --lang to 'full' when insightflow_synthetic_full.csv is ready

In [ ]:
# ── Step 5b : Fine-tune RoBERTa (EN specialist) ──────────────
!python ml/finetune.py --model roberta --lang en

In [ ]:
# ── Step 6 : Compare both models ─────────────────────────────
import json, os

print('='*60)
print('  Fine-tuning Results Comparison')
print('='*60)

for model_dir in ['insightflow-xlm-v1', 'insightflow-roberta-v1']:
    path = f'ml/models/{model_dir}/training_summary.json'
    if os.path.exists(path):
        with open(path) as f:
            s = json.load(f)
        print(f"\n  {s['model_name']}")
        print(f"  Base    : {s['model_base'].split('/')[-1]}")
        print(f"  Dataset : {s['dataset_size']} samples ({s['language']})")
        print(f"  Accuracy: {s['eval_accuracy']*100:.2f}%")
        print(f"  F1-macro: {s['eval_f1_macro']*100:.2f}%")
print('='*60)

In [ ]:
# ── Step 7 : Download fine-tuned models ──────────────────────
import shutil
from google.colab import files

for model_dir in ['insightflow-xlm-v1', 'insightflow-roberta-v1']:
    path = f'ml/models/{model_dir}'
    if os.path.exists(path):
        zip_path = f'{model_dir}.zip'
        shutil.make_archive(model_dir, 'zip', 'ml/models', model_dir)
        files.download(zip_path)
        print(f'Downloaded: {zip_path}')